In [ ]:
#파인콘
!pip install pinecone==6.0.2

In [3]:
from pinecone import Pinecone
import os

#환경 변수에서 Pinecone API 키 설정
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

In [5]:
from pinecone import ServerlessSpec

index_name="my-index"

pc.create_index(
    name=index_name,
    dimension=3, #벡터의 차원수, 삽입할 벡터의 크기와 동일해야함
    metric="cosine", #유사도 측정 방식(cosine, euclidean, dotproduct)
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-6vadili.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null
}

In [26]:
index = pc.Index(index_name)

#벡터 데이터 삽입
vectors = [
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6],
    [0.7, 0.8, 0.9]
]

#벡터를 인덱스에 추가
ids = ["doc1", "doc2", "doc3"]
#upsert:삽입하거나 기존 데이터 업데이트
index.upsert([(id, vector) for id, vector in zip(ids, vectors)])
print("아이템 임베딩을 인덱스에 추가헀습니다.")

import json

#유사 벡터 검색
results = index.query(
    vector=[[0.1, 0.2, 0.3]], #검색할 쿼리 벡터
    top_k=2,
    include_metadata=False
)

formatted_results = {
    "검색된 문서 ID": results["matches"][0]["id"],
    "유사도 거리": results["matches"][0]["score"]
}

print("\n유사한 벡터 검색 결과:")
print(json.dumps(formatted_results, indent=4, ensure_ascii=False))

아이템 임베딩을 인덱스에 추가헀습니다.

유사한 벡터 검색 결과:
{
    "검색된 문서 ID": "doc1",
    "유사도 거리": 0.998880625
}


In [31]:
#메타데이터와 함께 벡터 삽입
vectors = [
    ([0.1, 0.2, 0.3], {"category":"A", "year":2020}),
    ([0.4, 0.5, 0.6], {"category":"B", "year":2021}),
    ([0.7, 0.8, 0.9], {"category":"A", "year":2022}),
]

#벡터를 인덱스에 추가
ids = ["doc1", "doc2", "doc3"]
#upsert:삽입하거나 기존 데이터 업데이트
index.upsert([(id, vector, metadata) for id, (vector, metadata) in zip(ids, vectors)])

#검색 벡터
query_vector = [0.1, 0.2, 0.25]

#메터데이터 필터링 조건
filter_condition = {
    "category": {"$eq":"A"},
    "year": {"$gt":2020}
}

#검색
query_result = index.query(
    vector=query_vector,
    top_k=2,
    filter=filter_condition,
    include_metadata=True
)

print(query_result)   

formatted_results = {
    "검색된 문서 ID": query_result["matches"][0]["id"],
    "메타데이터": query_result["matches"][0]["metadata"]
}

print("\n유사한 벡터 검색 결과:")
print(json.dumps(formatted_results, indent=4, ensure_ascii=False))

{'matches': [{'id': 'doc3',
              'metadata': {'category': 'A', 'year': 2022.0},
              'score': 0.974112511,
              'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

유사한 벡터 검색 결과:
{
    "검색된 문서 ID": "doc3",
    "메타데이터": {
        "category": "A",
        "year": 2022.0
    }
}


In [33]:
#임베딩 기반 라마인덱스 답변 생성
!pip install llama-index==0.11.11
!pip install llama-index-vector-stores-pinecone==0.3.0
!pip install llama-index-embeddings-huggingface==0.3.0

In [39]:
from pinecone import Pinecone
from llama_index.core.schema import Document
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.schema import Document
import os

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)
index_name = "example"
spec = {
    "serverless":{
        "cloud":"aws",
        "region":"us-east-1"
    }
}

#인덱스 생성
index_lists = [item.get("name") for item in pc.list_indexes().get("indexes")]
if index_name not in index_lists:
    pc.create_index(
        name=index_name,
        dimension=3,
        metric="cosine",
        spec=spec
    )
index2 = pc.Index(index_name)

#llamaindex에서 사용할 huggingface 임베딩 모델 설정
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

#문서 데이터
documents = [
    "고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.",
    "강아지는 충성심이 강하고 친절한 동물로, 흔히 인간의 최고의 친구로 불립니다. 주로 애완동물로 기르고, 동반자로 유명합니다.",
    "고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다."
]

ids = ["doc1", "doc2", "doc3"]

#문서를 Document형식으로 변환
nodes = [Document(text=doc, id=doc_id) for doc, doc_id in zip(documents, ids)]

#Pinecone 벡터 스토어 생성
vector_store = PineconeVectorStore(pinecone_index=index2)

#llamaindex의 vectorstoreindex 생성
index2 = VectorStoreIndex.from_documents(nodes, vector_store=vector_store, embed_model=embed_model)

query_engine = index2.as_query_engine()
query_text = "고양이에 대해 알려줘"
response = query_engine.query(query_text)

print("\n[질의 결과]")
print(response)


[질의 결과]
고양이는 강아지와 마찬가지로 전 세계적으로 인기 있는 애완동물 중 하나입니다. 고양이는 독립적이고 까다로운 성격을 가지고 있으며, 주로 깨끗한 동물로 알려져 있습니다. 또한 사냥본능이 강하고 조용한 성격을 가지고 있어서 많은 사람들에게 사랑받고 있습니다.


In [40]:
#라마 인덱스 기반 답변 생성(임베딩 생략)
vector_store = PineconeVectorStore(pinecone_index=index2)

index2 = VectorStoreIndex.from_documents(nodes, vector_store=vector_store)
query_engine = index2.as_query_engine()
query_text = "고양이에 대해 알려줘"
response = query_engine.query(query_text)

print("\n[질의 결과]")
print(response)

#응답 생성에 사용된 문서 확인
print("\n[응답에 사용된 문서]")
for i, node in enumerate(response.source_nodes, 1):
    print(f"{i}. {node.text}\n")


[질의 결과]
고양이는 작은 육식동물로, 주로 애완동물로 기르며 민첩하고 장난기 있는 행동으로 유명합니다. 전 세계적으로 인기 있는 애완동물 중 하나이며, 강아지와는 다른 독특한 특징을 가지고 있습니다.

[응답에 사용된 문서]
1. 고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.

2. 고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다.

